In [1]:
from langchain_classic import text_splitter
#Data Ingestion
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from referencing import retrieval

loader = TextLoader("speech.txt")
text_documents = loader.load()
text_documents



C:\Users\Dhruv Patel\AppData\Local\Temp\ipykernel_12680\2111085417.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader


[Document(metadata={'source': 'speech.txt'}, page_content='Good morning everyone,\n\nToday I would like to talk about the importance of time.\n\nTime is one of the most valuable things in our lives.\nOnce time is lost, we can never get it back.\nEvery person gets the same twenty-four hours in a day.\nThe difference is how we choose to use those hours.\nStudents can use their time to learn new skills and gain knowledge.\nWorking professionals can use time to improve their careers.\nGood time management also reduces stress and confusion.\nPlanning our day helps us complete important tasks on time.\nWe should avoid spending too much time on unnecessary activities.\nTechnology can help us save time when we use it wisely.\nHowever, we should also make time for family, friends, and ourselves.\nTaking proper rest is equally important for a productive life.\nSmall actions performed consistently can produce great results.\nWe should focus on progress instead of waiting for the perfect moment.\n

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")



In [3]:
# Web based loader
from langchain_community.document_loaders import WebBaseLoader
import bs4

# load, chunk and index the content of the html page

loader=WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2026-07-04-harness/",),
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(
    class_=("post-title","post-content","post-header")))
)

text_documents = loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
text_documents

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2026-07-04-harness/'}, page_content='\n\n      Harness Engineering for Self-Improvement\n    \nDate: July 4, 2026  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nThe concept of recursive self-improvement (RSI) dates back to I. J. Good (1965), where he defined an “ultraintelligent machine” as a system that can surpass humans in all intellectual activities and design better machines to improve itself. Yudkowsky (2008) used the phrase “recursive self-improvement” for a specific feedback loop: an AI uses its current intelligence to improve the cognitive machinery that produces its intelligence.\nThis feedback loop in modern AI may indicate the model rewriting its own weights directly, or more broadly the model improves the training pipeline and the deployment system, which in turn enables a better successor model with improved performance across economically valuable tasks. The speed of research development i

In [5]:
# PDF reader
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('extended_attention_rag_testing.pdf')

docs=loader.load()

docs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-28T12:59:00+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-28T12:59:00+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'extended_attention_rag_testing.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='Extended Knowledge Document for RAG\n Testing\nAttention, Transformers, Embeddings, RAG and Sample Company Facts\n1. Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems capable of\nperforming tasks that normally require human intelligence. These tasks include language\nunderstanding, image recognition, reasoning, planning, prediction, and decision support. AI systems\ncan be rule-based or data-driven, although modern AI increasingly relies on machine learning.\nMachine learning is a branch of AI in whi

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents=text_splitter.split_documents(docs)
documents[:1]

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-28T12:59:00+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-28T12:59:00+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'extended_attention_rag_testing.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='Extended Knowledge Document for RAG\n Testing\nAttention, Transformers, Embeddings, RAG and Sample Company Facts\n1. Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems capable of\nperforming tasks that normally require human intelligence. These tasks include language\nunderstanding, image recognition, reasoning, planning, prediction, and decision support. AI systems\ncan be rule-based or data-driven, although modern AI increasingly relies on machine learning.\nMachine learning is a branch of AI in whi

In [8]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

print("Vector database created successfully!")

Vector database created successfully!


In [9]:
# CHROMA Vector database
query = "explain Word Embeddings"
result = db.similarity_search(query)
result[0].page_content

'An embedding is a numerical vector representing information in a continuous vector space. Text with\nsimilar meaning tends to have embeddings that are closer together. Embeddings can represent\nwords, sentences, paragraphs, images, products, or entire documents.\nSimilarity between embeddings is frequently measured using cosine similarity. Cosine similarity\ncompares the direction of two vectors. In retrieval systems, a query embedding can be compared\nagainst document embeddings to identify semantically related information.'

In [10]:
# FAISS Vector Database
from langchain_community.vectorstores import FAISS
db1=FAISS.from_documents(documents=documents, embedding=embeddings)


In [11]:
query1 = "Self-attention applies attention within a single sequence"
result = db1.similarity_search(query1)
result[0].page_content

"The model compares a Query with Keys to calculate relevance scores. These scores are usually\nnormalized with a softmax operation. The normalized scores become attention weights, which\ndetermine how strongly the corresponding Values contribute to the final output.\n6. Self-Attention\nSelf-attention applies attention within a single sequence. Each token can examine other tokens in the\nsame sequence to determine which ones provide useful context. This allows the model to capture\nrelationships between words regardless of their distance in the sequence.\nFor example, in the sentence 'The developer fixed the application because it had an error,'\nself-attention can help associate 'it' with 'application'. Such contextual relationships are important for\nlanguage understanding."

In [14]:
from langchain_community.llms import Ollama
# Load Ollama LAMA2 model
llm=Ollama(model="llama2")
llm

C:\Users\Dhruv Patel\AppData\Local\Temp\ipykernel_1904\2415645411.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm=Ollama(model="llama2")


Ollama(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}})

In [17]:
# Design ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context.
Think step by step before providing a detailed answer.
I will tip you $1000 if the user finds the answer helpful.
<context>
{context}
</context>
Question: {input}""")

In [22]:
# Chain Introduction
# Create Stuff Document Chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)

In [25]:
retrieval=db1.as_retriever()
retrieval

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002ABDA496D20>, search_kwargs={})

In [26]:
# Retrieval Chain
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retrieval,document_chain)

In [33]:
response=retrieval_chain.invoke({"input":"A RAG system should be evaluated at both the retrieval and generation stages."})

In [34]:
response['answer']

"Great, thank you for providing the context! To answer your question, a RAG (Retrieval-Augmented Generation) system should be evaluated at both the retrieval and generation stages.\n\nRetrieval evaluation:\nThe retrieval stage of a RAG system assesses whether the system found the correct source information in response to a user's query. This evaluation can be performed by checking whether the retrieved chunks contain the information the user was looking for. For example, if the user asked a question about NovaByte's remote work policy, the system should retrieve the relevant chunk(s) from the knowledge base and evaluate whether the retrieved information is accurate and relevant to the user's query.\n\nGeneration evaluation:\nThe generation stage of a RAG system evaluates whether the final response generated by the system is useful and relevant to the user's query. This evaluation can be performed by checking whether the generated answer addresses the user's question and provides the re